In [2]:
import pandas as pd

In [3]:
# import dataset
dataset = pd.read_csv("../dataset/cleaned_dataset.csv")
dataset.shape

(51093, 2)

In [4]:
dataset.head()

,processed_text,status
0,oh gosh,Anxiety
1,trouble sleeping confused mind restless heart ...,Anxiety
2,wrong back dear forward doubt stay restless re...,Anxiety
3,shifted focus something else still worried,Anxiety
4,restless restless month boy mean,Anxiety


In [5]:
have_missing_values = dataset['processed_text'].isnull().sum()
if have_missing_values != 0 : 
    dataset.dropna(inplace=True)

In [6]:
# column selections for X and y
X = dataset['processed_text'].values
y = dataset['status'].values

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [8]:
print(f"Total training samples {len(X_train)}")
print(f"Total testing samples {len(X_test)}")

Total training samples 40790
Total testing samples 10198


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# create the transformer
vectorizer = TfidfVectorizer()

# vectors
X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

In [10]:
from sklearn.metrics import classification_report

def print_message():
    print("Training complete")
    print("-" * 20)

def print_classification_report(y_test, y_pred, title=""):
    print(f"Classification Report: {title}")
    print(classification_report(y_test, y_pred, zero_division=0))   

def finalize_training(y_test, y_pred, title=""):
    print_message()
    print_classification_report(y_test, y_pred, title)


In [11]:
from sklearn.svm import LinearSVC

model = LinearSVC(C=1.0, random_state=42, class_weight='balanced')

In [12]:
# train model
model.fit(X_train_vector, y_train)

# prediction
y_pred = model.predict(X_test_vector)

finalize_training(y_test, y_pred, title="Linear SVC - Balanced")

Training complete
--------------------
Classification Report: Linear SVC - Balanced
                      precision    recall  f1-score   support

             Anxiety       0.71      0.79      0.75       725
             Bipolar       0.72      0.76      0.74       505
          Depression       0.71      0.62      0.66      3082
              Normal       0.88      0.92      0.90      3129
Personality disorder       0.45      0.51      0.47       178
              Stress       0.42      0.47      0.45       468
            Suicidal       0.63      0.64      0.64      2111

            accuracy                           0.73     10198
           macro avg       0.64      0.67      0.66     10198
        weighted avg       0.73      0.73      0.73     10198



## Handling Imbalance

### SMOTE

In [13]:
! pip install imbalanced-learn

In [14]:
from imblearn.over_sampling import SMOTE

In [15]:
smote = SMOTE(sampling_strategy="auto", random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train_vector, y_train)

print(f"Original train shape: {X_train_vector.shape}")
print(f"Resampled train shape: {X_train_resampled}")

Original train shape: (40790, 60384)
Resampled train shape:   (0, 24279)	0.17253707542824326
  (0, 43882)	0.17128870968605778
  (0, 22771)	0.5599645037704164
  (0, 18693)	0.16584752413821982
  (0, 30569)	0.3134319043075013
  (0, 2740)	0.12198525013959438
  (0, 21382)	0.10371077663230406
  (0, 10956)	0.172951754698956
  (0, 57975)	0.12877478813019227
  (0, 18704)	0.11660245221982793
  (0, 6774)	0.17578026774679048
  (0, 43040)	0.24667934135914665
  (0, 32319)	0.4165421691619379
  (0, 33820)	0.1950534881021047
  (0, 58460)	0.16419301275583978
  (0, 53534)	0.09247761458636979
  (0, 29292)	0.08634681554602612
  (0, 17226)	0.13151958794639573
  (0, 51266)	0.14490220229633083
  (0, 23425)	0.17690194178464783
  (1, 30569)	0.04093702389565996
  (1, 57975)	0.06727657914074149
  (1, 35892)	0.08326874092249374
  (1, 25812)	0.08494575406780885
  (1, 19515)	0.43950605251138647
  :	:
  (89668, 2636)	0.018595709143374675
  (89668, 43085)	0.014917175313794159
  (89668, 53534)	0.013779511257932388
  (8

In [16]:
# train model with resampled data
model = LinearSVC(C=1.5, random_state=42)

model.fit(X_train_resampled, y_train_resampled)

# prediction
y_pred = model.predict(X_test_vector)

finalize_training(y_test, y_pred, title="SVC with SMOTE")

y_pred = model.predict(vectorizer.transform([
    "I want to kill myself",
    "really worried, want to cry",
    "so frustrated so tired",
    "feel like restless"]))

y_pred

Training complete
--------------------
Classification Report: SVC with SMOTE
                      precision    recall  f1-score   support

             Anxiety       0.72      0.77      0.74       725
             Bipolar       0.71      0.75      0.73       505
          Depression       0.70      0.59      0.64      3082
              Normal       0.87      0.91      0.89      3129
Personality disorder       0.44      0.47      0.46       178
              Stress       0.38      0.45      0.41       468
            Suicidal       0.60      0.64      0.62      2111

            accuracy                           0.71     10198
           macro avg       0.63      0.65      0.64     10198
        weighted avg       0.71      0.71      0.71     10198



array(['Normal', 'Normal', 'Normal', 'Anxiety'], dtype=object)